<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/03_classical_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03 — Classical ML Baselines
**Proyecto:** Detección de Deslizamientos mediante Inteligencia Artificial  
**Institución:** Universidad EAFIT

Comparación sistemática de tres clasificadores clásicos como línea base antes de los modelos de aprendizaje profundo (notebooks 04–06). Todos los modelos comparten la misma extracción de características (HOG + estadísticas espectrales) y el mismo protocolo de validación cruzada estratificada (5-fold).

| Modelo | Complejidad | Característica clave |
|--------|-------------|---------------------|
| Logistic Regression | Lineal | Baseline mínimo interpretable |
| SVM (kernel RBF) | No lineal clásico | Margen máximo en espacio de características |
| Random Forest | Ensemble | Robusto, maneja no linealidades |

**Escalera de complejidad:** `LR (lineal) → SVM (no lineal) → RF (ensemble) → CNN (deep learning)`

![Vista previa: Classical Baselines — comparación LR vs SVM vs Random Forest](https://raw.githubusercontent.com/apmontesp/Landslides_-Applied-ML-Course/main/docs/figures/nb03_baseline_preview.png)

*Vista previa: Comparación de baselines clásicos — F1 por fold, curvas ROC y tabla comparativa*

In [ ]:
# ── Celda 1: Dependencias, Drive y carga de datos ──────────────────────────
from google.colab import drive
import h5py, json, warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from skimage.feature import hog

# Modelos clásicos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Validación y métricas
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, jaccard_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
)

warnings.filterwarnings('ignore')

# ── Montar Google Drive ────────────────────────────────────────────────────
drive.mount('/content/drive')

base_path = Path('/content/drive/MyDrive/Landslide4Sense')
img_dirs  = list(base_path.glob('**/TrainData/img'))

if img_dirs:
    train_img_dir  = img_dirs[0]
    train_mask_dir = train_img_dir.parent / 'mask'
    img_list       = sorted(list(train_img_dir.glob('*.h5')))
    mask_list      = sorted(list(train_mask_dir.glob('*.h5')))
    print(f'✅ {len(img_list)} parches detectados en Drive.')
else:
    raise FileNotFoundError('❌ Dataset no encontrado. Verifica la ruta MyDrive/Landslide4Sense/TrainData/img')

In [ ]:
# ── Celda 2: Extracción de características (HOG + estadísticas espectrales) ─
#
# Representación de cada parche 128×128×14:
#   • HOG sobre composición RGB falso color (canales 3,2,1 = Rojo, Verde, Azul S2)
#   • Pendiente media  (canal 12 — DEM ALOS)
#   • NDVI medio       (canal 3 NIR, canal 2 Rojo)
#   • Retrodispersión SAR VH media (canal 8)
#
# N_SAMPLES: SVM con kernel RBF escala O(n²) en tiempo de entrenamiento.
# 1 500 muestras dan un buen compromiso calidad/velocidad en Colab (~5 min SVM).

N_SAMPLES = 1500
X, y = [], []

for i in tqdm(range(min(N_SAMPLES, len(img_list))), desc='Extrayendo características'):
    with h5py.File(img_list[i], 'r') as hf:
        patch = hf[list(hf.keys())[0]][()]
    with h5py.File(mask_list[i], 'r') as hf:
        mask = hf[list(hf.keys())[0]][()]

    # HOG sobre RGB falso color
    rgb      = patch[:, :, [3, 2, 1]]
    rgb_norm = ((rgb - rgb.min()) / (np.ptp(rgb) + 1e-8) * 255).astype('uint8')
    feats_hog = hog(
        rgb_norm, pixels_per_cell=(16, 16),
        cells_per_block=(2, 2), channel_axis=-1, feature_vector=True
    )

    # Estadísticas espectrales adicionales
    slope   = np.mean(patch[:, :, 12])                     # pendiente DEM
    nir     = patch[:, :, 3];  red = patch[:, :, 2]
    ndvi    = np.mean((nir - red) / (nir + red + 1e-8))   # NDVI medio
    sar_vh  = np.mean(patch[:, :, 8])                      # SAR VH

    X.append(np.hstack([feats_hog, slope, ndvi, sar_vh]))
    y.append(int(np.max(mask) > 0))

X, y = np.array(X), np.array(y)
print(f'\nVector de características: {X.shape}')
print(f'Distribución de clases — Positivos: {y.sum()} ({100*y.mean():.1f}%) | Negativos: {(1-y).sum()} ({100*(1-y).mean():.1f}%)')

In [ ]:
# ── Celda 3: Baseline 1 — Logistic Regression ──────────────────────────────
#
# Modelo lineal. Sirve como cota inferior interpretable.
# Pipeline: StandardScaler → LR (necesario; LR es sensible a la escala).
# class_weight='balanced' compensa el desbalance de clases.

print('=' * 55)
print('  BASELINE 1 — LOGISTIC REGRESSION')
print('=' * 55)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, C=1.0,
                                   class_weight='balanced', random_state=42))
])

lr_records = []
lr_probs_all, lr_true_all = [], []

for fold, (t_idx, v_idx) in enumerate(skf.split(X, y)):
    lr_pipeline.fit(X[t_idx], y[t_idx])
    preds = lr_pipeline.predict(X[v_idx])
    probs = lr_pipeline.predict_proba(X[v_idx])[:, 1]

    f1   = f1_score(y[v_idx],   preds, zero_division=0)
    prec = precision_score(y[v_idx], preds, zero_division=0)
    rec  = recall_score(y[v_idx],   preds, zero_division=0)
    aucs = roc_auc_score(y[v_idx],  probs)
    iou  = jaccard_score(y[v_idx],  preds, zero_division=0)

    lr_records.append({'fold': fold+1, 'best_f1': f1, 'precision': prec,
                       'recall': rec, 'auc': aucs, 'iou': iou})
    lr_probs_all.extend(probs);  lr_true_all.extend(y[v_idx])
    print(f'  Fold {fold+1}: F1={f1:.4f} | AUC={aucs:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | IoU={iou:.4f}')

lr_mean_f1  = np.mean([r['best_f1']   for r in lr_records])
lr_std_f1   = np.std( [r['best_f1']   for r in lr_records])
lr_mean_auc = np.mean([r['auc']       for r in lr_records])
lr_mean_prec= np.mean([r['precision'] for r in lr_records])
lr_mean_rec = np.mean([r['recall']    for r in lr_records])
lr_mean_iou = np.mean([r['iou']       for r in lr_records])

print(f'\n  ✓ F1 medio: {lr_mean_f1:.4f} ± {lr_std_f1:.4f} | AUC: {lr_mean_auc:.4f}')
print('  → Interpretación: un modelo lineal puede capturar las señales más directas')
print('    (p. ej., elevación alta + vegetación dañada), pero pierde patrones espaciales.')

In [ ]:
# ── Celda 4: Baseline 2 — SVM con kernel RBF ───────────────────────────────
#
# Clasificador de margen máximo en espacio de características transformado.
# El kernel RBF permite capturar relaciones no lineales sin red profunda.
# Pipeline: StandardScaler → SVM (obligatorio; SVM muy sensible a escala).
# probability=True activa calibración de Platt para obtener probabilidades.
# Tiempo estimado: ~3–8 min con N_SAMPLES=1500 en Colab (CPU).

print('=' * 55)
print('  BASELINE 2 — SUPPORT VECTOR MACHINE (RBF)')
print('=' * 55)
print('  [Nota: SVM escala O(n²) — puede tomar 3-8 min]')

svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    SVC(kernel='rbf', C=1.0, gamma='scale',
                   probability=True, class_weight='balanced', random_state=42))
])

svm_records = []
svm_probs_all, svm_true_all = [], []

for fold, (t_idx, v_idx) in enumerate(skf.split(X, y)):
    svm_pipeline.fit(X[t_idx], y[t_idx])
    preds = svm_pipeline.predict(X[v_idx])
    probs = svm_pipeline.predict_proba(X[v_idx])[:, 1]

    f1   = f1_score(y[v_idx],   preds, zero_division=0)
    prec = precision_score(y[v_idx], preds, zero_division=0)
    rec  = recall_score(y[v_idx],   preds, zero_division=0)
    aucs = roc_auc_score(y[v_idx],  probs)
    iou  = jaccard_score(y[v_idx],  preds, zero_division=0)

    svm_records.append({'fold': fold+1, 'best_f1': f1, 'precision': prec,
                        'recall': rec, 'auc': aucs, 'iou': iou})
    svm_probs_all.extend(probs);  svm_true_all.extend(y[v_idx])
    print(f'  Fold {fold+1}: F1={f1:.4f} | AUC={aucs:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | IoU={iou:.4f}')

svm_mean_f1  = np.mean([r['best_f1']   for r in svm_records])
svm_std_f1   = np.std( [r['best_f1']   for r in svm_records])
svm_mean_auc = np.mean([r['auc']       for r in svm_records])
svm_mean_prec= np.mean([r['precision'] for r in svm_records])
svm_mean_rec = np.mean([r['recall']    for r in svm_records])
svm_mean_iou = np.mean([r['iou']       for r in svm_records])

print(f'\n  ✓ F1 medio: {svm_mean_f1:.4f} ± {svm_std_f1:.4f} | AUC: {svm_mean_auc:.4f}')
print('  → Interpretación: el kernel RBF captura relaciones no lineales entre canales')
print('    espectrales y topográficos. Buen candidato si los CNN no mejoran sustancialmente.')

In [ ]:
# ── Celda 5: Baseline 3 — Random Forest ────────────────────────────────────
#
# Ensemble de árboles de decisión. Robusto al ruido espectral y no requiere
# normalización previa. Ofrece importancia de variables como interpretabilidad.

print('=' * 55)
print('  BASELINE 3 — RANDOM FOREST')
print('=' * 55)

rf_model = RandomForestClassifier(n_estimators=100, n_jobs=-1,
                                   class_weight='balanced', random_state=42)

rf_records = []
rf_probs_all, rf_true_all = [], []
rf_last_model, rf_last_val_idx = None, None

for fold, (t_idx, v_idx) in enumerate(skf.split(X, y)):
    rf_model.fit(X[t_idx], y[t_idx])
    preds = rf_model.predict(X[v_idx])
    probs = rf_model.predict_proba(X[v_idx])[:, 1]

    f1   = f1_score(y[v_idx],   preds, zero_division=0)
    prec = precision_score(y[v_idx], preds, zero_division=0)
    rec  = recall_score(y[v_idx],   preds, zero_division=0)
    aucs = roc_auc_score(y[v_idx],  probs)
    iou  = jaccard_score(y[v_idx],  preds, zero_division=0)

    rf_records.append({'fold': fold+1, 'best_f1': f1, 'precision': prec,
                       'recall': rec, 'auc': aucs, 'iou': iou})
    rf_probs_all.extend(probs);  rf_true_all.extend(y[v_idx])
    rf_last_model, rf_last_val_idx = rf_model, v_idx
    print(f'  Fold {fold+1}: F1={f1:.4f} | AUC={aucs:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | IoU={iou:.4f}')

rf_mean_f1  = np.mean([r['best_f1']   for r in rf_records])
rf_std_f1   = np.std( [r['best_f1']   for r in rf_records])
rf_mean_auc = np.mean([r['auc']       for r in rf_records])
rf_mean_prec= np.mean([r['precision'] for r in rf_records])
rf_mean_rec = np.mean([r['recall']    for r in rf_records])
rf_mean_iou = np.mean([r['iou']       for r in rf_records])

print(f'\n  ✓ F1 medio: {rf_mean_f1:.4f} ± {rf_std_f1:.4f} | AUC: {rf_mean_auc:.4f}')
print('  → Interpretación: el ensemble reduce la varianza de los árboles individuales.')
print('    La importancia de variables revela qué bandas son más discriminativas.')

In [ ]:
# ── Celda 6: Tabla comparativa y visualizaciones ───────────────────────────

models_summary = [
    {'name': 'Logistic Regression', 'color': '#4e8df5',
     'f1': lr_mean_f1,   'std': lr_std_f1,   'auc': lr_mean_auc,
     'prec': lr_mean_prec, 'rec': lr_mean_rec, 'iou': lr_mean_iou,
     'probs': lr_probs_all, 'true': lr_true_all},
    {'name': 'SVM (RBF)',           'color': '#f5a623',
     'f1': svm_mean_f1,  'std': svm_std_f1,  'auc': svm_mean_auc,
     'prec': svm_mean_prec,'rec': svm_mean_rec,'iou': svm_mean_iou,
     'probs': svm_probs_all,'true': svm_true_all},
    {'name': 'Random Forest',       'color': '#2ecc71',
     'f1': rf_mean_f1,   'std': rf_std_f1,   'auc': rf_mean_auc,
     'prec': rf_mean_prec, 'rec': rf_mean_rec, 'iou': rf_mean_iou,
     'probs': rf_probs_all, 'true': rf_true_all},
]

# Tabla impresa
SEP = '-' * 75
print('\nCOMPARACIÓN DE BASELINES CLÁSICOS (5-Fold Cross-Validation)')
print(SEP)
print(f'{"Modelo":<22} {"F1 (media±std)":>18} {"AUC":>8} {"Precision":>10} {"Recall":>8} {"IoU":>8}')
print(SEP)
for m in models_summary:
    print(f"{m['name']:<22} {m['f1']:.4f} ± {m['std']:.4f}  {m['auc']:>8.4f} {m['prec']:>10.4f} {m['rec']:>8.4f} {m['iou']:>8.4f}")
print(SEP)

best = max(models_summary, key=lambda m: m['f1'])
print(f'\n→ Mejor baseline clásico: {best["name"]} (F1={best["f1"]:.4f})')
print('  Los modelos CNN (notebooks 04–06) deberán superar este umbral para justificar su complejidad.')

# ── Figura: F1 por modelo + Curvas ROC + Importancia RF ──
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# Panel 1: F1 con barras de error
ax = axes[0]
names  = [m['name'] for m in models_summary]
f1s    = [m['f1']   for m in models_summary]
stds   = [m['std']  for m in models_summary]
colors = [m['color'] for m in models_summary]
bars = ax.bar(names, f1s, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
ax.errorbar(names, f1s, yerr=stds, fmt='none', color='black', capsize=6, linewidth=2)
for bar, val in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.3f}',
            ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_ylabel('F1-Score (media 5-fold)', fontsize=12)
ax.set_title('F1-Score por Modelo', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', labelsize=10)

# Panel 2: Curvas ROC superpuestas
ax = axes[1]
for m in models_summary:
    fpr, tpr, _ = roc_curve(m['true'], m['probs'])
    roc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=m['color'], lw=2.5,
            label=f"{m['name']} (AUC={roc_val:.3f})")
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Aleatorio')
ax.set_xlabel('Tasa de Falsos Positivos', fontsize=11)
ax.set_ylabel('Tasa de Verdaderos Positivos', fontsize=11)
ax.set_title('Curvas ROC — Baselines Clásicos', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)

# Panel 3: Importancia de variables RF (top 15)
ax = axes[2]
importances = rf_last_model.feature_importances_
top_k = 15
indices = np.argsort(importances)[-top_k:]
feat_labels = []
for i in indices:
    if i == X.shape[1] - 3:
        feat_labels.append('Pendiente (DEM)')
    elif i == X.shape[1] - 2:
        feat_labels.append('NDVI medio')
    elif i == X.shape[1] - 1:
        feat_labels.append('SAR VH medio')
    else:
        feat_labels.append(f'HOG_{i}')
ax.barh(range(top_k), importances[indices], color='#2ecc71', alpha=0.85, edgecolor='white')
ax.set_yticks(range(top_k))
ax.set_yticklabels(feat_labels, fontsize=9)
ax.set_xlabel('Importancia (Gini)', fontsize=11)
ax.set_title('Top 15 Variables — Random Forest', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.suptitle('Classical ML Baselines — Landslide4Sense', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Celda 7: Guardar resultados en Drive ───────────────────────────────────
#
# Estructura de carpetas:
#   results/classical_baselines/
#     ├── comparison_summary.json      ← tabla comparativa completa
#     ├── logistic_regression/kfold_summary.json
#     ├── svm/kfold_summary.json
#     └── random_forest/kfold_summary.json  (también en results/random_forest/ para notebook 07)

base_results = Path('/content/drive/MyDrive/Landslide4Sense/results')

def save_model_results(folder, model_name, records, mean_f1, std_f1,
                        mean_auc, std_auc, mean_prec, mean_rec, mean_iou):
    folder.mkdir(parents=True, exist_ok=True)

    # v0 — compatible con notebook 07
    summary_v0 = {
        'model'  : model_name,
        'version': 0,
        'config' : {'n_samples': N_SAMPLES, 'n_folds': 5},
        'folds'  : [{'fold': r['fold'], 'best_f1': r['best_f1']} for r in records],
        'mean_f1': float(mean_f1),
        'std_f1' : float(std_f1),
    }
    with open(folder / 'kfold_summary.json', 'w') as f:
        json.dump(summary_v0, f, indent=2)

    # v2 — métricas completas
    summary_v2 = {
        'model'    : model_name,
        'version'  : 2,
        'config'   : {'n_samples': N_SAMPLES, 'n_folds': 5},
        'folds'    : records,
        'mean_f1'  : float(mean_f1),
        'std_f1'   : float(std_f1),
        'mean_auc' : float(mean_auc),
        'std_auc'  : float(std_auc),
        'mean_prec': float(mean_prec),
        'mean_rec' : float(mean_rec),
        'mean_iou' : float(mean_iou),
    }
    with open(folder / 'kfold_summary_v2.json', 'w') as f:
        json.dump(summary_v2, f, indent=2)

    print(f'  ✓ Guardado en: {folder}')


print('Guardando resultados en Drive...')

# Logistic Regression
save_model_results(
    base_results / 'classical_baselines' / 'logistic_regression',
    'logistic_regression', lr_records,
    lr_mean_f1, lr_std_f1, lr_mean_auc, np.std([r['auc'] for r in lr_records]),
    lr_mean_prec, lr_mean_rec, lr_mean_iou
)

# SVM
save_model_results(
    base_results / 'classical_baselines' / 'svm',
    'svm_rbf', svm_records,
    svm_mean_f1, svm_std_f1, svm_mean_auc, np.std([r['auc'] for r in svm_records]),
    svm_mean_prec, svm_mean_rec, svm_mean_iou
)

# Random Forest — doble guardado: carpeta propia + carpeta legacy para notebook 07
save_model_results(
    base_results / 'classical_baselines' / 'random_forest',
    'random_forest', rf_records,
    rf_mean_f1, rf_std_f1, rf_mean_auc, np.std([r['auc'] for r in rf_records]),
    rf_mean_prec, rf_mean_rec, rf_mean_iou
)
save_model_results(
    base_results / 'random_forest',       # legacy — notebook 07 lo lee aquí
    'random_forest', rf_records,
    rf_mean_f1, rf_std_f1, rf_mean_auc, np.std([r['auc'] for r in rf_records]),
    rf_mean_prec, rf_mean_rec, rf_mean_iou
)

# Tabla comparativa consolidada
comparison = {
    'notebook': '03_classical_baselines',
    'n_samples': N_SAMPLES,
    'n_folds'  : 5,
    'models'   : [
        {'name': m['name'], 'mean_f1': float(m['f1']), 'std_f1': float(m['std']),
         'mean_auc': float(m['auc']), 'mean_prec': float(m['prec']),
         'mean_rec': float(m['rec']), 'mean_iou': float(m['iou'])}
        for m in models_summary
    ],
    'best_classical': best['name'],
    'best_f1'       : float(best['f1']),
}
cb_dir = base_results / 'classical_baselines'
cb_dir.mkdir(parents=True, exist_ok=True)
with open(cb_dir / 'comparison_summary.json', 'w') as f:
    json.dump(comparison, f, indent=2)

# Guardar figura comparativa
fig.savefig(cb_dir / 'classical_baselines_comparison.png', dpi=150, bbox_inches='tight')
print('  ✓ Figura guardada: classical_baselines_comparison.png')

print('\nRESUMEN FINAL — CLASSICAL BASELINES')
SEP = '-' * 55
print(SEP)
for m in models_summary:
    print(f"  {m['name']:<22}: F1={m['f1']:.4f} ± {m['std']:.4f} | AUC={m['auc']:.4f}")
print(SEP)
print(f'  Mejor baseline : {best["name"]} (F1={best["f1"]:.4f})')
print('  Umbral a superar por modelos CNN (notebooks 04–06).')
print(SEP)